In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ak11105/spam-phishing-dataset/splits/splits/val.jsonl
/kaggle/input/datasets/ak11105/spam-phishing-dataset/splits/splits/test.jsonl
/kaggle/input/datasets/ak11105/spam-phishing-dataset/splits/splits/train.jsonl
/kaggle/input/datasets/ak11105/spam-phishing-dataset/splits/splits/train_augmented.jsonl
/kaggle/input/datasets/ak11105/spam-phishing-dataset/splits/splits/dataset.jsonl
/kaggle/input/datasets/ak11105/spam-phishing-dataset/final/final/val.jsonl
/kaggle/input/datasets/ak11105/spam-phishing-dataset/final/final/test.jsonl
/kaggle/input/datasets/ak11105/spam-phishing-dataset/final/final/train.jsonl


In [2]:
!pip install -q wandb pyyaml

import os, json, math, random, re
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import RobertaTokenizerFast, RobertaModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report

import wandb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [3]:
from kaggle_secrets import UserSecretsClient
secret_label = "wandb_api"
secret_value = UserSecretsClient().get_secret(secret_label)

In [4]:
wandb.login(key=secret_value)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ak9248 (ak9248-scratchmind) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
CFG = {
    'run_name': 'roberta-multimodal-final',
    'text_encoder': 'roberta-base',
    'metadata_dim': 10,
    'fusion_hidden': 256,
    'dropout': 0.3,
    'num_classes': 3,
    'max_length': 256,
    'epochs': 3,
    'batch_size': 16,
    'lr': 2e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'class_weights': [1.0, 1.0, 2.0],

    'metadata_dropout': 0.4,
    'metadata_noise': 0.1,
    'mode': 'full',

    'train': '/kaggle/input/datasets/ak11105/spam-phishing-dataset/final/final/train.jsonl',
    'val':   '/kaggle/input/datasets/ak11105/spam-phishing-dataset/final/final/val.jsonl',
    'test':  '/kaggle/input/datasets/ak11105/spam-phishing-dataset/final/final/test.jsonl',
}

wandb.init(project="email-triage", name=CFG['run_name'], config=CFG)

LABEL2ID = {'spam': 0, 'junk': 1, 'phishing': 2}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
URL_REGEX = re.compile(r'https?://\S+|www\.\S+')

def normalize_text(text):
    return URL_REGEX.sub("<URL>", text or "")

In [7]:
SPF_MAP  = {'pass': 1.0, 'softfail': 0.5, 'fail': 0.0, 'none': -1.0}
AUTH_MAP = {'pass': 1.0, 'fail': 0.0, 'none': -1.0}

def encode_metadata(r):
    return np.array([
        SPF_MAP.get(r.get('spf_result','none'), -1.0),
        AUTH_MAP.get(r.get('dkim_result','none'), -1.0),
        AUTH_MAP.get(r.get('dmarc_result','none'), -1.0),
        r.get('url_count',0.0),
        r.get('attachment_count',0.0),
        float(r.get('reply_to_mismatch',False)),
        r.get('html_text_ratio',0.0),
        r.get('tld_risk_score',1.0),
        float(r.get('sender_seen_before',False)),
        float(r.get('first_time_domain',True)),
    ], dtype=np.float32)

In [8]:
class EmailDataset(Dataset):
    def __init__(self, path, tokenizer):
        self.data = []
        texts, metas, labels = [], [], []

        sep = tokenizer.sep_token

        for line in open(path):
            r = json.loads(line)
            if r.get('label') not in LABEL2ID:
                continue

            text = (
                normalize_text(r.get('subject')) + f' {sep} ' +
                normalize_text(r.get('sender_display_name')) + f' {sep} ' +
                normalize_text(r.get('url_token_text')) + f' {sep} ' +
                normalize_text(r.get('body_text'))
            )

            texts.append(text)
            metas.append(encode_metadata(r))
            labels.append(LABEL2ID[r['label']])

        enc = tokenizer(texts, max_length=CFG['max_length'],
                        truncation=True, padding='max_length')

        for i in range(len(texts)):
            self.data.append({
                'input_ids': torch.tensor(enc['input_ids'][i]),
                'attention_mask': torch.tensor(enc['attention_mask'][i]),
                'metadata': torch.tensor(metas[i]),
                'label': torch.tensor(labels[i])
            })

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

In [9]:
tokenizer = RobertaTokenizerFast.from_pretrained(CFG['text_encoder'])

train_ds = EmailDataset(CFG['train'], tokenizer)
val_ds   = EmailDataset(CFG['val'], tokenizer)
test_ds  = EmailDataset(CFG['test'], tokenizer)

labels = [x['label'].item() for x in train_ds]
weights = 1. / np.bincount(labels)
sample_weights = [weights[l] for l in labels]

train_loader = DataLoader(
    train_ds,
    batch_size=CFG['batch_size'],
    sampler=WeightedRandomSampler(sample_weights, len(sample_weights)),
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'])
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'])

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(CFG['text_encoder'])
        self.meta = nn.Sequential(
            nn.Linear(10,64),
            nn.ReLU(),
            nn.LayerNorm(64),
            nn.Dropout(0.3)
        )
        self.fc = nn.Linear(768+64,3)

    def pool(self, h, m):
        m = m.unsqueeze(-1)
        return (h*m).sum(1)/m.sum(1)

    def forward(self, ids, mask, meta):
        if CFG['mode']=="text_only":
            meta = torch.zeros_like(meta)
        if CFG['mode']=="metadata_only":
            ids = torch.zeros_like(ids)

        h = self.encoder(ids, attention_mask=mask).last_hidden_state
        h = self.pool(h, mask)
        meta = self.meta(meta)

        return self.fc(torch.cat([h,meta],1))

model = Model().to(device)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(CFG['class_weights']).to(device)
)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'])

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    int(len(train_loader)*CFG['epochs']*CFG['warmup_ratio']),
    len(train_loader)*CFG['epochs']
)

In [12]:
def apply_metadata(meta):
    if torch.rand(1) < CFG['metadata_dropout']:
        meta = torch.zeros_like(meta)

    for i in [5,8,9]:
        mask = torch.rand(meta.size(0)) < CFG['metadata_noise']
        meta[mask, i] = torch.abs(meta[mask, i] - 1)

    return meta

In [13]:
def compute_trust(logits):
    probs = torch.softmax(logits, dim=-1)
    max_prob, preds = probs.max(dim=-1)

    top2 = torch.topk(probs, 2, dim=-1).values
    margin = top2[:, 0] - top2[:, 1]

    trust = 0.6 * max_prob + 0.4 * margin
    return preds, trust, probs

In [14]:
def run_eval(loader, prefix, epoch):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for b in loader:
            logits = model(
                b['input_ids'].to(device),
                b['attention_mask'].to(device),
                b['metadata'].to(device)
            )
            preds.extend(logits.argmax(-1).cpu().tolist())
            labels.extend(b['label'].tolist())

    report = classification_report(
        labels, preds,
        target_names=['spam','junk','phishing'],
        output_dict=True,
        zero_division=0
    )

    wandb.log({
        f"{prefix}/accuracy": report['accuracy'],
        f"{prefix}/phishing_recall": report['phishing']['recall'],
        f"{prefix}/macro_f1": report['macro avg']['f1-score'],
        "epoch": epoch
    })

    print(f"\n{prefix.upper()}")
    print(classification_report(labels, preds, target_names=['spam','junk','phishing']))

    return report

In [15]:
scaler = torch.amp.GradScaler(device='cuda')
best = 0

for epoch in range(CFG['epochs']):
    model.train()
    total_loss = 0

    for b in tqdm(train_loader):
        optimizer.zero_grad()

        meta = apply_metadata(b['metadata'].to(device))

        with torch.amp.autocast('cuda'):
            logits = model(
                b['input_ids'].to(device),
                b['attention_mask'].to(device),
                meta
            )
            loss = criterion(logits, b['label'].to(device))

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss/len(train_loader)

    wandb.log({
        "train/loss": avg_loss,
        "epoch": epoch
    })

    val_report = run_eval(val_loader, "val", epoch)

    if val_report['phishing']['recall'] > best:
        best = val_report['phishing']['recall']
        torch.save(model.state_dict(),"best.pt")

# 🔥 FINAL TEST EVAL
run_eval(test_loader, "test", epoch)

  0%|          | 0/2399 [00:00<?, ?it/s]/tmp/ipykernel_57/994954793.py:24: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
100%|██████████| 2399/2399 [08:37<00:00,  4.64it/s]



VAL
              precision    recall  f1-score   support

        spam       0.99      0.99      0.99      2995
        junk       1.00      1.00      1.00      2316
    phishing       0.99      0.99      0.99      3115

    accuracy                           0.99      8426
   macro avg       0.99      0.99      0.99      8426
weighted avg       0.99      0.99      0.99      8426



100%|██████████| 2399/2399 [08:39<00:00,  4.61it/s]



VAL
              precision    recall  f1-score   support

        spam       1.00      0.99      0.99      2995
        junk       1.00      1.00      1.00      2316
    phishing       0.99      1.00      0.99      3115

    accuracy                           1.00      8426
   macro avg       1.00      1.00      1.00      8426
weighted avg       1.00      1.00      1.00      8426



100%|██████████| 2399/2399 [08:39<00:00,  4.62it/s]



VAL
              precision    recall  f1-score   support

        spam       1.00      0.99      0.99      2995
        junk       1.00      1.00      1.00      2316
    phishing       0.99      1.00      0.99      3115

    accuracy                           1.00      8426
   macro avg       1.00      1.00      1.00      8426
weighted avg       1.00      1.00      1.00      8426


TEST
              precision    recall  f1-score   support

        spam       1.00      0.99      0.99      3004
        junk       1.00      1.00      1.00      2298
    phishing       0.99      1.00      0.99      2834

    accuracy                           0.99      8136
   macro avg       1.00      1.00      1.00      8136
weighted avg       0.99      0.99      0.99      8136



{'spam': {'precision': 0.9959812458137978,
  'recall': 0.9900133155792277,
  'f1-score': 0.9929883138564274,
  'support': 3004.0},
 'junk': {'precision': 0.9995650282731623,
  'recall': 1.0,
  'f1-score': 0.999782466826191,
  'support': 2298.0},
 'phishing': {'precision': 0.9894773763591722,
  'recall': 0.9954128440366973,
  'f1-score': 0.9924362357080035,
  'support': 2834.0},
 'accuracy': 0.9947148475909537,
 'macro avg': {'precision': 0.9950078834820442,
  'recall': 0.9951420532053082,
  'f1-score': 0.9950690054635406,
  'support': 8136.0},
 'weighted avg': {'precision': 0.9947279968041137,
  'recall': 0.9947148475909537,
  'f1-score': 0.9947150068323226,
  'support': 8136.0}}